# T08 — Đo lệch miền điện thoại → phim

Notebook chỉ đánh giá artifact A đã đóng băng; không train, không đọc `movie_train.csv`, không chọn siêu tham số trên test. Seed cố định: `42`. Bảng chính dùng duy nhất checkpoint research canonical `ml/artifacts/phobert_A_best/`.

In [ ]:
from pathlib import Path
import hashlib, json, math, os, random, sys
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/firefly-cinema')]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / 'ml' / 'src' / 'eval_utils.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'ml' / 'src'))
from eval_utils import compute_metrics, normalize
from backend.app.classifier import classifier as backend_classifier, normalize as backend_normalize

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('seed =', SEED)

## Gate dữ liệu, artifact và `normalize()`

Nhãn `1 = tiêu cực`, `0 = bình thường`. Cell này import trực tiếp `normalize()` từ source backend và so sánh với bản contract trong `eval_utils.py` trên 20 câu (10 phone + 10 movie).

In [ ]:
PHONE_PATH = PROJECT_ROOT / 'data' / 'uit-visfd' / 'processed' / 'test.csv'
MOVIE_PATH = PROJECT_ROOT / 'data' / 'phim' / 'movie_test.csv'
RESEARCH_CKPT = PROJECT_ROOT / 'ml' / 'artifacts' / 'phobert_A_best'
BACKEND_CKPT = PROJECT_ROOT / 'backend' / 'models' / 'phobert-binhluan'
BASELINE_PATHS = {
    'SVM A': PROJECT_ROOT / 'ml' / 'artifacts' / 'baselines' / 'svm_A.joblib',
    'Logistic Regression A': PROJECT_ROOT / 'ml' / 'artifacts' / 'baselines' / 'lr_A.joblib',
    'Naive Bayes A': PROJECT_ROOT / 'ml' / 'artifacts' / 'baselines' / 'nb_A.joblib',
}
for path in [PHONE_PATH, MOVIE_PATH, RESEARCH_CKPT / 'model.safetensors', BACKEND_CKPT / 'model.safetensors', *BASELINE_PATHS.values()]:
    assert path.exists(), f'Missing required artifact: {path}'

phone = pd.read_csv(PHONE_PATH)
movie = pd.read_csv(MOVIE_PATH)
assert phone.shape[0] == 2224 and phone['has_negative'].value_counts().sort_index().to_dict() == {0: 981, 1: 1243}
assert list(movie.columns) == ['id', 'text', 'label', 'source_type', 'note']
assert movie.shape[0] == 149 and movie['label'].value_counts().sort_index().to_dict() == {0: 48, 1: 101}
assert not phone[['text_raw', 'has_negative']].isna().any().any()
assert not movie[['id', 'text', 'label']].isna().any().any()

parity_texts = phone['text_raw'].astype(str).head(10).tolist() + movie['text'].astype(str).head(10).tolist()
parity = pd.DataFrame({'text': parity_texts})
parity['backend'] = parity['text'].map(backend_normalize)
parity['eval_utils'] = parity['text'].map(normalize)
parity['equal'] = parity['backend'] == parity['eval_utils']
assert len(parity) == 20 and parity['equal'].all()
display(parity)
print('AC-01_NORMALIZE_PARITY_OK=20/20')
print('phone_sha256 =', hashlib.sha256(PHONE_PATH.read_bytes()).hexdigest())
print('movie_sha256 =', hashlib.sha256(MOVIE_PATH.read_bytes()).hexdigest())
print('research_checkpoint_sha256 =', hashlib.sha256((RESEARCH_CKPT / 'model.safetensors').read_bytes()).hexdigest())
print('backend_checkpoint_sha256 =', hashlib.sha256((BACKEND_CKPT / 'model.safetensors').read_bytes()).hexdigest())

## Đánh giá hai miền

Công thức: `delta_f1_neg = F1_neg(phone) - F1_neg(test_set)`. Do đó phone có delta `0`; delta movie dương nghĩa là F1 giảm khi chuyển sang miền phim, delta âm nghĩa là F1 phim cao hơn.

In [ ]:
phone_eval = pd.DataFrame({'id': phone['index'], 'text': phone['text_raw'].astype(str), 'label': phone['has_negative'].astype(int)})
movie_eval = movie[['id', 'text', 'label']].copy()
datasets = {'phone': phone_eval, 'movie': movie_eval}
for frame in datasets.values():
    frame['input'] = frame['text'].map(normalize)

predictions, scores = {}, {}
baseline_models = {name: joblib.load(path) for name, path in BASELINE_PATHS.items()}
for model_name, model in baseline_models.items():
    for test_set, frame in datasets.items():
        texts = frame['input'].tolist()
        predictions[(model_name, test_set)] = np.asarray(model.predict(texts), dtype=int)
        if hasattr(model, 'predict_proba'):
            scores[(model_name, test_set)] = np.asarray(model.predict_proba(texts))[:, list(model.classes_).index(1)]
        else:
            decision = np.asarray(model.decision_function(texts), dtype=float)
            scores[(model_name, test_set)] = np.where(decision >= 0, 1/(1+np.exp(-decision)), np.exp(decision)/(1+np.exp(decision)))

tokenizer = AutoTokenizer.from_pretrained(RESEARCH_CKPT, local_files_only=True)
research_model = AutoModelForSequenceClassification.from_pretrained(RESEARCH_CKPT, local_files_only=True)
research_model.eval()
for test_set, frame in datasets.items():
    negative_scores = []
    with torch.no_grad():
        for start in range(0, len(frame), 16):
            inputs = tokenizer(frame['input'].iloc[start:start+16].tolist(), padding=True, truncation=True, max_length=256, return_tensors='pt')
            negative_scores.extend(torch.softmax(research_model(**inputs).logits, dim=-1)[:, 1].cpu().tolist())
    scores[('PhoBERT A', test_set)] = np.asarray(negative_scores)
    predictions[('PhoBERT A', test_set)] = (scores[('PhoBERT A', test_set)] >= 0.5).astype(int)

model_order = ['PhoBERT A', 'SVM A', 'Logistic Regression A', 'Naive Bayes A']
rows = []
for model_name in model_order:
    phone_f1 = compute_metrics(datasets['phone']['label'], predictions[(model_name, 'phone')])['f1_neg']
    for test_set in ['phone', 'movie']:
        metrics = compute_metrics(datasets[test_set]['label'], predictions[(model_name, test_set)])
        rows.append({'model': model_name, 'test_set': test_set, **{key: metrics[key] for key in ['precision_neg','recall_neg','f1_neg','macro_f1']}, 'delta_f1_neg': phone_f1 - metrics['f1_neg']})
result = pd.DataFrame(rows)
comparison = pd.read_csv(PROJECT_ROOT / 'ml' / 'results' / 'comparison_A.csv').query("input == 'normalize'").copy()
comparison['model'] = comparison['model'].replace({'SVM':'SVM A','Logistic Regression':'Logistic Regression A','Naive Bayes':'Naive Bayes A'})
phone_rows = result.query("test_set == 'phone'").set_index('model')
comparison = comparison.set_index('model')
parity_diffs = {(model, metric): abs(float(phone_rows.loc[model, metric]) - float(comparison.loc[model, metric])) for model in model_order for metric in ['precision_neg','recall_neg','f1_neg','macro_f1']}
assert max(parity_diffs.values()) <= 0.001
assert result.shape == (8, 7)
RESULT_PATH = PROJECT_ROOT / 'ml' / 'results' / 'domain_shift.csv'
result.round(6).to_csv(RESULT_PATH, index=False)
display(result.round(6))
print('AC-02_ROWS_OK=8/8')
print('AC-03_PHONE_PARITY_MAX_DIFF=', max(parity_diffs.values()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
pivot = result.pivot(index='model', columns='test_set', values='f1_neg').loc[model_order]
x = np.arange(len(model_order)); width = 0.36
ax.bar(x-width/2, pivot['phone'], width, label='Phone (UIT-ViSFD)', color='#2f6fed')
ax.bar(x+width/2, pivot['movie'], width, label='Movie', color='#f59e0b')
for xpos, value in zip(x-width/2, pivot['phone']): ax.text(xpos, value+0.008, f'{value:.3f}', ha='center', fontsize=9)
for xpos, value in zip(x+width/2, pivot['movie']): ax.text(xpos, value+0.008, f'{value:.3f}', ha='center', fontsize=9)
ax.set_xticks(x, ['PhoBERT A','SVM A','LR A','NB A']); ax.set_ylim(0, 1.02)
ax.set_ylabel('F1 lớp tiêu cực'); ax.set_title('F1 tiêu cực: miền điện thoại và phim')
ax.grid(axis='y', alpha=.25); ax.legend(); fig.tight_layout()
FIGURE_PATH = PROJECT_ROOT / 'ml' / 'results' / 'figures' / 'domain_shift_f1.png'
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True); fig.savefig(FIGURE_PATH, dpi=180); plt.show()
small_shift = result.query("test_set == 'movie' and delta_f1_neg < 0.03")
print('WARNING_DELTA_LT_0.03:', small_shift['model'].tolist() if len(small_shift) else 'none')

## Câu kiểm tra bắt buộc và 20 lỗi phim

Câu bắt buộc không nằm trong `movie_test.csv`, nên được đánh giá như ca kiểm thử riêng. Score là xác suất/score quy đổi của nhãn 1; ngưỡng `0,5`.

In [ ]:
SPECIAL = 'Phim dài lê thê, xem mà mệt'
special_input = normalize(SPECIAL)
special_rows = []
with torch.no_grad():
    special_research_score = float(torch.softmax(research_model(**tokenizer(special_input, return_tensors='pt', truncation=True, max_length=256)).logits, dim=-1)[0,1])
special_rows.append({'model':'PhoBERT A research canonical','prediction':int(special_research_score >= .5),'score_neg':special_research_score})
for model_name, model in baseline_models.items():
    prediction = int(model.predict([special_input])[0])
    if hasattr(model, 'predict_proba'):
        score = float(model.predict_proba([special_input])[0][list(model.classes_).index(1)])
    else:
        decision = float(model.decision_function([special_input])[0]); score = 1/(1+math.exp(-decision)) if decision >= 0 else math.exp(decision)/(1+math.exp(decision))
    special_rows.append({'model':model_name,'prediction':prediction,'score_neg':score})
backend_prediction, backend_score = backend_classifier._models['phobert'].predict(SPECIAL)
special_rows.append({'model':'PhoBERT A backend (tham chiếu, không dùng trong bảng chính)','prediction':int(backend_prediction),'score_neg':backend_score})
special = pd.DataFrame(special_rows)
display(special)
assert float(special.query("model == 'PhoBERT A research canonical'")['score_neg'].iloc[0]) < .5
assert int(special.iloc[-1]['prediction']) == 0

selected = {
 'MOV0502': ('Từ vựng đặc thù phim', '“khóc trôi kem nền” biểu đạt cảm xúc tích cực theo ngữ cảnh xem phim.'),
 'MOV0476': ('Từ vựng đặc thù phim', 'Tiếng lóng nhấn mạnh “vl” đi cùng “hay” bị NB đọc sai.'),
 'MOV0479': ('Từ vựng đặc thù phim', 'Cụm ngắn “ảo vãi” mơ hồ sắc thái; cả bốn model bỏ sót nhãn tiêu cực.'),
 'MOV0207': ('Từ vựng đặc thù phim', 'Chê phim ca nhạc bằng câu rất ngắn; PhoBERT không nhận ra hàm ý.'),
 'MOV0199': ('Từ vựng đặc thù phim', '“Phản diện đáng quên” là phê bình đặc thù phim, ba baseline bỏ sót.'),
 'MOV0503': ('Chê nhịp phim/nội dung', '“Dễ đoán” và “không đáng sợ” là chê nội dung nhưng PhoBERT dự đoán bình thường.'),
 'MOV0432': ('Chê nhịp phim/nội dung', 'Nhiều mệnh đề khen/chê trộn lẫn làm PhoBERT nghiêng về bình thường.'),
 'MOV0484': ('Chê nhịp phim/nội dung', '“Dài dòng lê thê” là lỗi nhịp phim điển hình nhưng cả bốn model bỏ sót.'),
 'MOV0171': ('Chê nhịp phim/nội dung', 'Chê tình tiết/kĩ xảo/diễn xuất mức nhẹ; ba baseline dự đoán bình thường.'),
 'MOV0411': ('Chê nhịp phim/nội dung', 'Cấu trúc góp ý “nếu... thì hay hơn” che khuất phê bình nhịp phim.'),
 'MOV0403': ('Chê nhịp phim/nội dung', '“Hời hời hợt” là cụm chê ngắn đặc thù, PhoBERT cho score rất thấp.'),
 'MOV0448': ('Châm biếm', 'So sánh rạp với “lò thiêu” và câu hỏi tu từ; SVM nằm sát dưới ngưỡng.'),
 'MOV0405': ('Châm biếm', '“Điểm cộng duy nhất” hàm ý phần còn lại kém; cả bốn model bỏ sót.'),
 'MOV0060': ('Châm biếm', '“Cảnh đẹp thì oke chứ hay gì” là phủ định mỉa mai; ba model bỏ sót.'),
 'MOV0229': ('Châm biếm', 'Kể chuyện khán giả cười thay vì khóc để chê gián tiếp; PhoBERT bỏ sót.'),
 'MOV0303': ('Chê rạp hoặc dịch vụ', 'Review pha trộn: rạp bé nhưng giá/âm thanh tốt; PhoBERT báo dương tính giả.'),
 'MOV0310': ('Chê rạp hoặc dịch vụ', 'Chê cơ sở/lobby nhưng khen phòng chiếu và giá; NB bỏ sót nhãn tiêu cực.'),
 'MOV0297': ('Chê rạp hoặc dịch vụ', 'Phàn nàn nhân viên/combo/giá; ba baseline bỏ sót.'),
 'MOV0301': ('Chê rạp hoặc dịch vụ', 'Chê thời gian phục vụ và số quầy; cả bốn model bỏ sót.'),
 'MOV0312': ('Chê rạp hoặc dịch vụ', 'Phàn nàn upsell bắp nước; PhoBERT bỏ sót do từ vựng dịch vụ.'),
}
assert len(selected) == 20 and set(selected).issubset(set(movie_eval['id']))
error_rows = []
for item_id, (category, reason) in selected.items():
    idx = movie_eval.index[movie_eval['id'] == item_id][0]; row = movie_eval.loc[idx]
    model_values, wrong = {}, []
    for model_name in model_order:
        pred = int(predictions[(model_name, 'movie')][idx]); score = float(scores[(model_name, 'movie')][idx])
        model_values[model_name] = f'{pred} ({score:.6f})'
        if pred != int(row['label']): wrong.append(model_name)
    assert wrong
    error_rows.append({'id':item_id,'category':category,'truth':int(row['label']),'text':row['text'],'wrong_models':', '.join(wrong),'predictions':model_values,'reason':reason})
print('AC-04_SPECIAL_SENTENCE_OK')
print('AC-05_REPRESENTATIVE_ERRORS_OK=', len(error_rows))

In [ ]:
lines = [
 '# Phân tích lỗi T08 — lệch miền điện thoại → phim', '',
 '## Phạm vi và quy ước', '',
 '- Bảng chính dùng checkpoint research canonical `ml/artifacts/phobert_A_best/`; checkpoint backend chỉ đối chiếu câu bắt buộc.',
 '- Nhãn `0 = bình thường`, `1 = tiêu cực`; score trong ngoặc là score nhãn 1; ngưỡng `0,5`.',
 '- Hai score PhoBERT khác nhau vì hai checkpoint khác trọng số; không trộn chúng trong bảng metric chính.', '',
 '## Câu kiểm tra bắt buộc', '', f'`{SPECIAL}` không có trong `movie_test.csv`, nên đây là ca kiểm thử riêng.', '',
 '| Model/checkpoint | Dự đoán | Score tiêu cực |', '|---|---:|---:|',
 *[f"| {row.model} | {int(row.prediction)} | {float(row.score_neg):.6f} |" for row in special.itertuples()], '',
 'Cả research và backend PhoBERT đều dự đoán `0`, nhưng checkpoint research có score cao hơn rõ rệt. Kết quả này cho thấy cách chê nhịp phim “dài lê thê” chưa được hai checkpoint nhận ra đủ mạnh.', '',
 '## 20 lỗi tiêu biểu trên movie test', '',
 '| # | ID | Nhóm | Nhãn thật | Model sai | Dự đoán/score (PhoBERT; SVM; LR; NB) | Câu | Phân tích |',
 '|---:|---|---|---:|---|---|---|---|',
]
for number, item in enumerate(error_rows, 1):
    values = item['predictions']; pred_text = '; '.join([f"{name}: {values[name]}" for name in model_order])
    clean_text = str(item['text']).replace('|', '&#124;').replace('\n',' ')
    lines.append(f"| {number} | {item['id']} | {item['category']} | {item['truth']} | {item['wrong_models']} | {pred_text} | {clean_text} | {item['reason']} |")
lines += ['', '## Tổng hợp', '']
counts = pd.Series([item['category'] for item in error_rows]).value_counts()
lines += [f'- {category}: **{int(count)}** câu.' for category, count in counts.items()]
lines += ['', 'Mẫu lỗi cho thấy lệch miền không chỉ ở từ khóa: các model còn khó xử lý phê bình nhẹ/pha trộn, hàm ý châm biếm và từ vựng dịch vụ rạp. Đây là phân tích mô tả trên 20 ca được chọn có chủ đích, không phải ước lượng tần suất cho toàn bộ lỗi.']
ERROR_PATH = PROJECT_ROOT / 'ml' / 'results' / 'domain_shift_errors.md'
ERROR_PATH.write_text('\n'.join(lines) + '\n', encoding='utf-8')
assert ERROR_PATH.read_text(encoding='utf-8').count('| MOV') == 20
print('NOTEBOOK_EXECUTION_OK')